# ResMill Interactive Explorer — Alluvsim Edition

Full Alluvsim-faithful channel engine with all six facies (FF, FFCH, CS, LV, LA, CH).

**New in this notebook:**
* All ~50 Alluvsim parameters exposed for `MeanderingChannelLayer` and `BraidedChannelLayer`.
* **Preset selector**: PV-shoestring, CB-jigsaw, CB-labyrinth, SH-distal, SH-proximal — apply Alluvsim's canonical reservoir-type defaults with one click.
* **Output facies mode**: `'binary'` (0=shale, 1=sand) for dataset use, or `'alluvsim'` (full -1..4 colour-coded categorical) to inspect architectural elements.
* `Lobe`, `Gaussian`, `Delta` panels are unchanged (those layers don't have Alluvsim equivalents).

Adjust sliders and the plot regenerates automatically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display, clear_output
import resmill as gr
from resmill.layers.channel import (
    PV_SHOESTRING, CB_JIGSAW, CB_LABYRINTH, SH_DISTAL, SH_PROXIMAL,
    FACIES_PROPS,
)

%matplotlib inline

# Full 6-class colour palette for output_facies='alluvsim' mode
ALLUVSIM_FACIES_NAMES = {
    -1: 'FF (floodplain)', 0: 'FFCH (mud plug)', 1: 'CS (splay)',
    2: 'LV (levee)', 3: 'LA (point bar)', 4: 'CH (channel)',
}
ALLUVSIM_FACIES_COLORS = {
    -1: '#b4b4b4', 0: '#5b4636', 1: '#f2d16b',
    2: '#e8a23a', 3: '#c89b5e', 4: '#7a3f14',
}

def alluvsim_cmap():
    codes = sorted(ALLUVSIM_FACIES_COLORS.keys())
    cmap = mcolors.ListedColormap([ALLUVSIM_FACIES_COLORS[c] for c in codes])
    bounds = [c - 0.5 for c in codes] + [codes[-1] + 0.5]
    return cmap, mcolors.BoundaryNorm(bounds, cmap.N)

# ── Pre-warm numba JIT so the first user-triggered Generate is fast ──
# Without this, the first click pays a ~2.5s numba compile cost.
# This runs ONCE at notebook startup (~2-3s); afterwards every channel
# generation is ~0.5s on a typical 80×50×20 grid.
print('Pre-compiling numba kernels (one-time, ~3s)...')
_warmup_layer = gr.MeanderingChannelLayer(
    nx=30, ny=20, nz=10, x_len=300, y_len=200, z_len=5, top_depth=0)
_warmup_layer.create_geology(
    output_facies='alluvsim', seed=0, ntime=10,
    nlevel=2, level_z=[1.0, 3.0], NTGtarget=0.05,
    mLVdepth=0.5, mLVwidth=20, mLVheight=0.4,
    mCSnum=1, mCSnumlobe=1,
)
del _warmup_layer
print('Ready! Generate clicks will now be fast.')

## Model Builder

1. Choose a model type from the dropdown
2. (Channel models) Pick a preset to load Alluvsim's canonical parameters, OR keep `Custom` and tweak individually
3. **Drag sliders** to set parameters — nothing renders yet
4. Click the **Generate** button (blue) to run the simulation and render the plot
5. Or tick **Auto-regenerate** to make every slider change render immediately (good for single-parameter sweeps)

Picking a preset always renders immediately. The Generate button always renders. The auto-toggle only affects whether *individual slider drags* trigger re-runs.

In [ ]:
# ── Shared style + grid widgets ───────────────────────────────────────
style = {'description_width': '180px'}
layout = widgets.Layout(width='430px')

w_nx = widgets.IntSlider(value=80, min=20, max=200, step=10, description='nx', style=style, layout=layout)
w_ny = widgets.IntSlider(value=50, min=20, max=200, step=10, description='ny', style=style, layout=layout)
w_nz = widgets.IntSlider(value=20, min=5, max=80, step=5, description='nz', style=style, layout=layout)
w_xlen = widgets.IntSlider(value=800, min=200, max=5000, step=100, description='x_len (m)', style=style, layout=layout)
w_ylen = widgets.IntSlider(value=500, min=200, max=5000, step=100, description='y_len (m)', style=style, layout=layout)
w_zlen = widgets.IntSlider(value=10, min=4, max=300, step=2, description='z_len (m)', style=style, layout=layout)
w_depth = widgets.IntSlider(value=0, min=0, max=8000, step=100, description='top_depth (m)', style=style, layout=layout)
w_dip = widgets.FloatSlider(value=0.0, min=0.0, max=10.0, step=0.5, description='dip (deg)', style=style, layout=layout)
w_seed = widgets.IntText(value=69069, description='seed', style=style, layout=layout)

grid_box = widgets.VBox([
    widgets.HTML('<b>Grid</b>'),
    widgets.HBox([widgets.VBox([w_nx, w_ny, w_nz, w_depth, w_seed]),
                  widgets.VBox([w_xlen, w_ylen, w_zlen, w_dip])])
])

# View options
w_view = widgets.Dropdown(
    options=['Cube Slices', 'Z Slices', 'X Slices', 'Y Slices'],
    value='Cube Slices', description='View', style=style, layout=layout)
w_prop = widgets.Dropdown(
    options=['Porosity', 'Permeability', 'Facies (binary)', 'Facies (Alluvsim 6-class)'],
    value='Facies (Alluvsim 6-class)', description='Property', style=style, layout=layout)
view_box = widgets.VBox([widgets.HTML('<b>Visualization</b>'), widgets.HBox([w_view, w_prop])])

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Lobe / Gaussian / Delta widgets — unchanged from interactive.ipynb
# (no Alluvsim equivalent for these layers).
# ──────────────────────────────────────────────────────────────────

# Lobe
l_poro = widgets.FloatSlider(value=0.20, min=0.05, max=0.40, step=0.01, description='poro_ave', style=style, layout=layout)
l_perm = widgets.FloatSlider(value=1.5, min=0.1, max=5.0, step=0.1, description='perm_ave (log10 mD)', style=style, layout=layout)
l_poro_std = widgets.FloatSlider(value=0.03, min=0.005, max=0.10, step=0.005, description='poro_std', style=style, layout=layout)
l_perm_std = widgets.FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05, description='perm_std', style=style, layout=layout)
l_ntg = widgets.FloatSlider(value=0.7, min=0.1, max=1.0, step=0.05, description='NTG', style=style, layout=layout)
l_dhmin = widgets.FloatSlider(value=4.0, min=1.0, max=10.0, step=0.5, description='dhmin', style=style, layout=layout)
l_dhmax = widgets.FloatSlider(value=4.0, min=1.0, max=15.0, step=0.5, description='dhmax', style=style, layout=layout)
l_rmin = widgets.IntSlider(value=15, min=5, max=50, step=1, description='rmin', style=style, layout=layout)
l_rmax = widgets.IntSlider(value=25, min=10, max=60, step=1, description='rmax', style=style, layout=layout)
l_asp = widgets.FloatSlider(value=1.5, min=1.0, max=5.0, step=0.1, description='aspect', style=style, layout=layout)
l_theta = widgets.FloatSlider(value=0.0, min=-90.0, max=90.0, step=5.0, description='theta0', style=style, layout=layout)
l_m = widgets.IntSlider(value=100, min=10, max=500, step=10, description='m', style=style, layout=layout)
l_upthin = widgets.Checkbox(value=False, description='upthinning', style=style, layout=layout)
l_bouma = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description='bouma_factor', style=style, layout=layout)
lobe_box = widgets.VBox([widgets.HTML('<b>Lobe Parameters</b>'),
    widgets.HBox([widgets.VBox([l_poro, l_perm, l_poro_std, l_perm_std, l_ntg, l_bouma, l_theta]),
                  widgets.VBox([l_dhmin, l_dhmax, l_rmin, l_rmax, l_asp, l_m, l_upthin])])])

# Gaussian
g_poro = widgets.FloatSlider(value=0.15, min=0.05, max=0.40, step=0.01, description='poro_ave', style=style, layout=layout)
g_perm = widgets.FloatSlider(value=1.2, min=0.1, max=5.0, step=0.1, description='perm_ave (log10 mD)', style=style, layout=layout)
g_poro_std = widgets.FloatSlider(value=0.03, min=0.005, max=0.10, step=0.005, description='poro_std', style=style, layout=layout)
g_perm_std = widgets.FloatSlider(value=0.4, min=0.05, max=2.0, step=0.05, description='perm_std', style=style, layout=layout)
g_ntg = widgets.FloatSlider(value=0.6, min=0.1, max=1.0, step=0.05, description='NTG', style=style, layout=layout)
g_fx = widgets.FloatSlider(value=2.5, min=0.5, max=10.0, step=0.5, description='facies_filt X', style=style, layout=layout)
g_fy = widgets.FloatSlider(value=5.0, min=0.5, max=15.0, step=0.5, description='facies_filt Y', style=style, layout=layout)
g_fz = widgets.FloatSlider(value=2.5, min=0.5, max=10.0, step=0.5, description='facies_filt Z', style=style, layout=layout)
g_sx = widgets.FloatSlider(value=1.5, min=0.5, max=5.0, step=0.5, description='sand_filt X', style=style, layout=layout)
g_sy = widgets.FloatSlider(value=2.5, min=0.5, max=8.0, step=0.5, description='sand_filt Y', style=style, layout=layout)
g_sz = widgets.FloatSlider(value=1.5, min=0.5, max=5.0, step=0.5, description='sand_filt Z', style=style, layout=layout)
g_nugget = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description='nugget', style=style, layout=layout)
g_corr = widgets.FloatSlider(value=0.6, min=0.0, max=1.0, step=0.05, description='poro_perm_corr', style=style, layout=layout)
gauss_box = widgets.VBox([widgets.HTML('<b>Gaussian Parameters</b>'),
    widgets.HBox([widgets.VBox([g_poro, g_perm, g_poro_std, g_perm_std, g_ntg, g_nugget, g_corr]),
                  widgets.VBox([g_fx, g_fy, g_fz, g_sx, g_sy, g_sz])])])

# Delta
d_feeder = widgets.FloatSlider(value=60.0, min=10.0, max=300.0, step=5.0, description='feeder_width', style=style, layout=layout)
d_ngen = widgets.IntSlider(value=8, min=1, max=30, step=1, description='n_generations', style=style, layout=layout)
d_fan = widgets.FloatSlider(value=95.0, min=20.0, max=180.0, step=5.0, description='fan_angle_deg', style=style, layout=layout)
d_bif = widgets.IntSlider(value=4, min=1, max=6, step=1, description='bifurcation_depth', style=style, layout=layout)
d_kmin = widgets.IntSlider(value=3, min=2, max=6, step=1, description='children_min', style=style, layout=layout)
d_kmax = widgets.IntSlider(value=5, min=2, max=8, step=1, description='children_max', style=style, layout=layout)
d_asym = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.05, description='fan_asymmetry', style=style, layout=layout)
d_azim = widgets.FloatSlider(value=0.0, min=0.0, max=360.0, step=5.0, description='azimuth (deg)', style=style, layout=layout)
d_apex = widgets.FloatSlider(value=0.25, min=0.05, max=0.8, step=0.05, description='apex_x_fraction', style=style, layout=layout)
d_prog = widgets.FloatSlider(value=0.0, min=0.0, max=0.6, step=0.05, description='progradation_frac', style=style, layout=layout)
d_trunk = widgets.FloatSlider(value=5.5, min=1.0, max=15.0, step=0.5, description='trunk_length_factor', style=style, layout=layout)
d_ltap = widgets.FloatSlider(value=0.88, min=0.5, max=1.0, step=0.02, description='length_taper', style=style, layout=layout)
d_wtap = widgets.FloatSlider(value=0.80, min=0.3, max=1.0, step=0.02, description='width_taper', style=style, layout=layout)
d_mwig = widgets.FloatSlider(value=0.04, min=0.0, max=0.2, step=0.01, description='meander_amp', style=style, layout=layout)
d_mbL = widgets.FloatSlider(value=2.2, min=0.5, max=6.0, step=0.1, description='mb_length_factor', style=style, layout=layout)
d_mbW = widgets.FloatSlider(value=1.6, min=0.5, max=5.0, step=0.1, description='mb_width_factor', style=style, layout=layout)
d_mbhw = widgets.FloatSlider(value=0.06, min=0.01, max=0.3, step=0.01, description='mb_hw_ratio', style=style, layout=layout)
d_mbdw = widgets.FloatSlider(value=0.08, min=0.01, max=0.3, step=0.01, description='mb_dw_ratio', style=style, layout=layout)
d_dwr = widgets.FloatSlider(value=0.25, min=0.05, max=1.0, step=0.05, description='dwratio', style=style, layout=layout)
d_poro = widgets.FloatSlider(value=0.25, min=0.05, max=0.40, step=0.01, description='poro_ave', style=style, layout=layout)
delta_box = widgets.VBox([widgets.HTML('<b>Delta Parameters</b>'),
    widgets.HBox([widgets.VBox([d_feeder, d_ngen, d_fan, d_bif, d_kmin, d_kmax, d_asym, d_azim, d_apex, d_prog, d_dwr]),
                  widgets.VBox([d_trunk, d_ltap, d_wtap, d_mwig, d_mbL, d_mbW, d_mbhw, d_mbdw, d_poro])])])

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Channel widgets — full Alluvsim parameter set
# Used by both MeanderingChannelLayer and BraidedChannelLayer
# (same engine; Braided just defaults to CB-jigsaw values).
# ──────────────────────────────────────────────────────────────────

# --- Aggradation
ch_nlevel = widgets.IntSlider(value=5, min=1, max=12, step=1, description='nlevel', style=style, layout=layout)
ch_lvl_z = widgets.Text(value='1.5, 3.0, 5.0, 7.0, 9.0', description='level_z (csv, m)', style=style, layout=layout)
ch_NTG = widgets.FloatSlider(value=0.10, min=0.02, max=0.80, step=0.02, description='NTGtarget', style=style, layout=layout)
ch_ntime = widgets.IntSlider(value=120, min=20, max=500, step=10, description='ntime (max events)', style=style, layout=layout)

# --- Avulsion
ch_pAvOut = widgets.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.01, description='probAvulOutside', style=style, layout=layout)
ch_pAvIn = widgets.FloatSlider(value=0.05, min=0.0, max=1.0, step=0.01, description='probAvulInside', style=style, layout=layout)

# --- Channel geometry (per-event Gaussian draws)
ch_mDepth = widgets.FloatSlider(value=2.5, min=0.5, max=12.0, step=0.1, description='mCHdepth', style=style, layout=layout)
ch_sDepth = widgets.FloatSlider(value=0.3, min=0.0, max=2.0, step=0.05, description='stdevCHdepth', style=style, layout=layout)
ch_sDepth2 = widgets.FloatSlider(value=0.2, min=0.0, max=1.0, step=0.05, description='stdevCHdepth2', style=style, layout=layout)
ch_mWdr = widgets.FloatSlider(value=10.0, min=2.0, max=40.0, step=0.5, description='mCHwdratio (W/D)', style=style, layout=layout)
ch_sWdr = widgets.FloatSlider(value=1.0, min=0.0, max=8.0, step=0.1, description='stdevCHwdratio', style=style, layout=layout)
ch_mSinu = widgets.FloatSlider(value=1.6, min=1.0, max=2.0, step=0.05, description='mCHsinu', style=style, layout=layout)
ch_sSinu = widgets.FloatSlider(value=0.15, min=0.0, max=0.5, step=0.05, description='stdevCHsinu', style=style, layout=layout)
ch_mAzi = widgets.FloatSlider(value=90.0, min=0.0, max=360.0, step=5.0, description='mCHazi', style=style, layout=layout)
ch_sAzi = widgets.FloatSlider(value=1.0, min=0.0, max=30.0, step=0.5, description='stdevCHazi', style=style, layout=layout)
ch_mSrc = widgets.FloatSlider(value=250.0, min=0.0, max=1000.0, step=10.0, description='mCHsource (Y, m)', style=style, layout=layout)
ch_sSrc = widgets.FloatSlider(value=80.0, min=0.0, max=300.0, step=10.0, description='stdevCHsource', style=style, layout=layout)

# --- Migration
ch_mMig = widgets.FloatSlider(value=35.0, min=5.0, max=200.0, step=5.0, description='mdistMigrate (m)', style=style, layout=layout)
ch_sMig = widgets.FloatSlider(value=10.0, min=0.0, max=80.0, step=2.0, description='stdevdistMigrate', style=style, layout=layout)

# --- Levee (LV)
ch_mLVd = widgets.FloatSlider(value=0.6, min=0.0, max=4.0, step=0.1, description='mLVdepth', style=style, layout=layout)
ch_sLVd = widgets.FloatSlider(value=0.1, min=0.0, max=1.0, step=0.05, description='stdevLVdepth', style=style, layout=layout)
ch_mLVw = widgets.FloatSlider(value=40.0, min=0.0, max=400.0, step=5.0, description='mLVwidth', style=style, layout=layout)
ch_sLVw = widgets.FloatSlider(value=5.0, min=0.0, max=80.0, step=1.0, description='stdevLVwidth', style=style, layout=layout)
ch_mLVh = widgets.FloatSlider(value=0.4, min=0.0, max=3.0, step=0.1, description='mLVheight', style=style, layout=layout)
ch_sLVh = widgets.FloatSlider(value=0.1, min=0.0, max=1.0, step=0.05, description='stdevLVheight', style=style, layout=layout)
ch_mLVa = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05, description='mLVasym', style=style, layout=layout)
ch_sLVa = widgets.FloatSlider(value=0.1, min=0.0, max=0.5, step=0.05, description='stdevLVasym', style=style, layout=layout)
ch_mLVt = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description='mLVthin', style=style, layout=layout)
ch_sLVt = widgets.FloatSlider(value=0.0, min=0.0, max=0.5, step=0.05, description='stdevLVthin', style=style, layout=layout)

# --- Crevasse splay (CS) — default off (mCSnum=0)
ch_mCSn = widgets.FloatSlider(value=0.0, min=0.0, max=10.0, step=0.5, description='mCSnum', style=style, layout=layout)
ch_sCSn = widgets.FloatSlider(value=0.0, min=0.0, max=5.0, step=0.5, description='stdevCSnum', style=style, layout=layout)
ch_mCSnl = widgets.FloatSlider(value=0.0, min=0.0, max=10.0, step=0.5, description='mCSnumlobe', style=style, layout=layout)
ch_sCSnl = widgets.FloatSlider(value=0.0, min=0.0, max=5.0, step=0.5, description='stdevCSnumlobe', style=style, layout=layout)
ch_mCSs = widgets.FloatSlider(value=50.0, min=0.0, max=200.0, step=5.0, description='mCSsource', style=style, layout=layout)
ch_sCSs = widgets.FloatSlider(value=20.0, min=0.0, max=100.0, step=5.0, description='stdevCSsource', style=style, layout=layout)
ch_mCSLL = widgets.FloatSlider(value=200.0, min=20.0, max=2000.0, step=20.0, description='mCSLOLL', style=style, layout=layout)
ch_sCSLL = widgets.FloatSlider(value=50.0, min=0.0, max=500.0, step=10.0, description='stdevCSLOLL', style=style, layout=layout)
ch_mCSWW = widgets.FloatSlider(value=30.0, min=5.0, max=500.0, step=5.0, description='mCSLOWW', style=style, layout=layout)
ch_sCSWW = widgets.FloatSlider(value=10.0, min=0.0, max=100.0, step=5.0, description='stdevCSLOWW', style=style, layout=layout)
ch_mCSl = widgets.FloatSlider(value=100.0, min=10.0, max=1000.0, step=10.0, description='mCSLOl', style=style, layout=layout)
ch_sCSl = widgets.FloatSlider(value=20.0, min=0.0, max=200.0, step=5.0, description='stdevCSLOl', style=style, layout=layout)
ch_mCSw = widgets.FloatSlider(value=20.0, min=2.0, max=200.0, step=2.0, description='mCSLOw', style=style, layout=layout)
ch_sCSw = widgets.FloatSlider(value=10.0, min=0.0, max=80.0, step=2.0, description='stdevCSLOw', style=style, layout=layout)
ch_mCShw = widgets.FloatSlider(value=0.03, min=0.001, max=0.5, step=0.01, description='mCSLO_hwratio', style=style, layout=layout)
ch_sCShw = widgets.FloatSlider(value=0.01, min=0.0, max=0.2, step=0.005, description='stdevCSLO_hwratio', style=style, layout=layout)
ch_mCSdw = widgets.FloatSlider(value=0.02, min=0.001, max=0.5, step=0.01, description='mCSLO_dwratio', style=style, layout=layout)
ch_sCSdw = widgets.FloatSlider(value=0.005, min=0.0, max=0.2, step=0.005, description='stdevCSLO_dwratio', style=style, layout=layout)

# --- Abandoned-channel mud plug (FFCH)
ch_mFFCH = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description='mFFCHprop', style=style, layout=layout)
ch_sFFCH = widgets.FloatSlider(value=0.0, min=0.0, max=0.3, step=0.02, description='stdevFFCHprop', style=style, layout=layout)

# --- Hydraulic
ch_Cf = widgets.FloatLogSlider(value=0.0036, base=10, min=-4, max=-1, step=0.05, description='Cf (friction)', style=style, layout=layout, readout_format='.4f')
ch_scour = widgets.FloatSlider(value=10.0, min=0.5, max=30.0, step=0.5, description='scour_factor', style=style, layout=layout)
ch_grad = widgets.FloatLogSlider(value=0.001, base=10, min=-4, max=-1, step=0.05, description='gradient', style=style, layout=layout, readout_format='.4f')
ch_Q = widgets.FloatSlider(value=5.0, min=0.5, max=50.0, step=0.5, description='Q (discharge)', style=style, layout=layout)

# --- Pool / discretisation
ch_CHndraw = widgets.IntSlider(value=50, min=10, max=200, step=10, description='CHndraw', style=style, layout=layout)
ch_ndiscr = widgets.IntSlider(value=10, min=5, max=50, step=1, description='ndiscr', style=style, layout=layout)
ch_nCHcor = widgets.IntSlider(value=10, min=2, max=50, step=1, description='nCHcor', style=style, layout=layout)

# --- Presentation
ch_azim = widgets.FloatSlider(value=0.0, min=0.0, max=360.0, step=5.0, description='azimuth (deg)', style=style, layout=layout)
ch_outmode = widgets.Dropdown(options=['binary', 'alluvsim'], value='alluvsim', description='output_facies', style=style, layout=layout)

channel_box = widgets.VBox([
    widgets.HTML('<b>Channel Parameters — Aggradation, Avulsion, Geometry</b>'),
    widgets.HBox([
        widgets.VBox([ch_nlevel, ch_lvl_z, ch_NTG, ch_ntime, ch_pAvOut, ch_pAvIn,
                      ch_mDepth, ch_sDepth, ch_sDepth2, ch_mWdr, ch_sWdr, ch_mSinu, ch_sSinu,
                      ch_mAzi, ch_sAzi, ch_mSrc, ch_sSrc]),
        widgets.VBox([ch_mMig, ch_sMig,
                      ch_mLVd, ch_sLVd, ch_mLVw, ch_sLVw, ch_mLVh, ch_sLVh, ch_mLVa, ch_sLVa, ch_mLVt, ch_sLVt,
                      ch_mFFCH, ch_sFFCH, ch_Cf, ch_scour, ch_grad]),
    ]),
    widgets.HTML('<b>Crevasse Splay (set mCSnum &gt; 0 to enable)</b>'),
    widgets.HBox([
        widgets.VBox([ch_mCSn, ch_sCSn, ch_mCSnl, ch_sCSnl, ch_mCSs, ch_sCSs,
                      ch_mCSLL, ch_sCSLL, ch_mCSWW, ch_sCSWW]),
        widgets.VBox([ch_mCSl, ch_sCSl, ch_mCSw, ch_sCSw,
                      ch_mCShw, ch_sCShw, ch_mCSdw, ch_sCSdw])
    ]),
    widgets.HTML('<b>Pool / Discretisation / Output</b>'),
    widgets.HBox([
        widgets.VBox([ch_Q, ch_CHndraw, ch_ndiscr, ch_nCHcor]),
        widgets.VBox([ch_azim, ch_outmode])
    ])
])

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Preset selector — applies Alluvsim canonical parameter dicts
# ──────────────────────────────────────────────────────────────────
PRESETS = {
    'PV Shoestring': PV_SHOESTRING,
    'CB Jigsaw': CB_JIGSAW,
    'CB Labyrinth': CB_LABYRINTH,
    'SH Distal': SH_DISTAL,
    'SH Proximal': SH_PROXIMAL,
}
PRESET_KW_TO_WIDGET = {
    'nlevel': ch_nlevel, 'NTGtarget': ch_NTG, 'ntime': ch_ntime,
    'probAvulOutside': ch_pAvOut, 'probAvulInside': ch_pAvIn,
    'mCHdepth': ch_mDepth, 'stdevCHdepth': ch_sDepth, 'stdevCHdepth2': ch_sDepth2,
    'mCHwdratio': ch_mWdr, 'stdevCHwdratio': ch_sWdr,
    'mCHsinu': ch_mSinu, 'stdevCHsinu': ch_sSinu,
    'mCHazi': ch_mAzi, 'stdevCHazi': ch_sAzi,
    'mCHsource': ch_mSrc, 'stdevCHsource': ch_sSrc,
    'mdistMigrate': ch_mMig, 'stdevdistMigrate': ch_sMig,
    'mLVdepth': ch_mLVd, 'stdevLVdepth': ch_sLVd,
    'mLVwidth': ch_mLVw, 'stdevLVwidth': ch_sLVw,
    'mLVheight': ch_mLVh, 'stdevLVheight': ch_sLVh,
    'mLVasym': ch_mLVa, 'stdevLVasym': ch_sLVa,
    'mLVthin': ch_mLVt, 'stdevLVthin': ch_sLVt,
    'mCSnum': ch_mCSn, 'stdevCSnum': ch_sCSn,
    'mCSnumlobe': ch_mCSnl, 'stdevCSnumlobe': ch_sCSnl,
    'mCSsource': ch_mCSs, 'stdevCSsource': ch_sCSs,
    'mCSLOLL': ch_mCSLL, 'stdevCSLOLL': ch_sCSLL,
    'mCSLOWW': ch_mCSWW, 'stdevCSLOWW': ch_sCSWW,
    'mCSLOl': ch_mCSl, 'stdevCSLOl': ch_sCSl,
    'mCSLOw': ch_mCSw, 'stdevCSLOw': ch_sCSw,
    'mCSLO_hwratio': ch_mCShw, 'stdevCSLO_hwratio': ch_sCShw,
    'mCSLO_dwratio': ch_mCSdw, 'stdevCSLO_dwratio': ch_sCSdw,
    'mFFCHprop': ch_mFFCH, 'stdevFFCHprop': ch_sFFCH,
    'Cf': ch_Cf, 'scour_factor': ch_scour, 'gradient': ch_grad, 'Q': ch_Q,
    'CHndraw': ch_CHndraw, 'ndiscr': ch_ndiscr, 'nCHcor': ch_nCHcor,
}

preset_selector = widgets.Dropdown(
    options=['Custom'] + list(PRESETS.keys()),
    value='Custom', description='Preset', style=style, layout=layout)

_applying_preset = {'lock': False}

def apply_preset(*args):
    name = preset_selector.value
    if name == 'Custom':
        return
    _applying_preset['lock'] = True
    try:
        preset = PRESETS[name]
        for kw, val in preset.items():
            if kw == 'level_z':
                ch_lvl_z.value = ', '.join(f'{v:g}' for v in val)
                continue
            w = PRESET_KW_TO_WIDGET.get(kw)
            if w is None:
                continue
            try:
                w.value = val
            except Exception:
                pass  # silently skip if value out of slider range
    finally:
        _applying_preset['lock'] = False
    # Picking a preset always regenerates, regardless of the auto-mode toggle
    generate_model(force=True)

preset_selector.observe(apply_preset, names='value')
preset_box = widgets.VBox([widgets.HTML('<b>Channel Preset (loads canonical Alluvsim params)</b>'), preset_selector])

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Model selector + dynamic panel + manual/auto generate
# ──────────────────────────────────────────────────────────────────
import time

model_selector = widgets.Dropdown(
    options=['Lobe', 'Gaussian', 'Meandering Channel', 'Braided Channel', 'Delta'],
    value='Meandering Channel', description='Model Type', style=style, layout=layout)

# ── Generate button + auto-regenerate toggle + status label ──────
# Default auto OFF: dragging sliders is smooth; click "Generate" to render.
w_auto = widgets.Checkbox(
    value=False,
    description='Auto-regenerate on slider change',
    style={'description_width': 'initial'})
w_gen_btn = widgets.Button(
    description='Generate', button_style='primary', icon='play',
    layout=widgets.Layout(width='180px'))
status_label = widgets.HTML(value='<i style="color:#666">Click Generate to render the first model.</i>')
gen_box = widgets.VBox([widgets.HBox([w_gen_btn, w_auto]), status_label])

param_area = widgets.Output()
# Output area with a fixed min-height + border so VS Code keeps the slot
# visible even between renders (works around the widget-Output update
# flickering on VS Code's Jupyter extension).
output_area = widgets.Output(
    layout=widgets.Layout(min_height='620px', border='1px solid #d0d0d0', padding='4px'))

param_panels = {
    'Lobe': lobe_box,
    'Gaussian': gauss_box,
    'Meandering Channel': widgets.VBox([preset_box, channel_box]),
    'Braided Channel': widgets.VBox([preset_box, channel_box]),
    'Delta': delta_box,
}

def update_params(*args):
    with param_area:
        clear_output(wait=True)
        display(param_panels[model_selector.value])

model_selector.observe(update_params, names='value')

def _channel_kwargs():
    """Collect all channel widgets into a kwargs dict for create_geology."""
    try:
        level_z = [float(s.strip()) for s in ch_lvl_z.value.split(',') if s.strip()]
    except ValueError:
        level_z = list(np.linspace(w_zlen.value / max(ch_nlevel.value, 1), w_zlen.value, ch_nlevel.value))
    return dict(
        nlevel=ch_nlevel.value, level_z=level_z,
        NTGtarget=ch_NTG.value, ntime=ch_ntime.value,
        probAvulOutside=ch_pAvOut.value, probAvulInside=ch_pAvIn.value,
        mCHdepth=ch_mDepth.value, stdevCHdepth=ch_sDepth.value, stdevCHdepth2=ch_sDepth2.value,
        mCHwdratio=ch_mWdr.value, stdevCHwdratio=ch_sWdr.value,
        mCHsinu=ch_mSinu.value, stdevCHsinu=ch_sSinu.value,
        mCHazi=ch_mAzi.value, stdevCHazi=ch_sAzi.value,
        mCHsource=ch_mSrc.value, stdevCHsource=ch_sSrc.value,
        mdistMigrate=ch_mMig.value, stdevdistMigrate=ch_sMig.value,
        mLVdepth=ch_mLVd.value, stdevLVdepth=ch_sLVd.value,
        mLVwidth=ch_mLVw.value, stdevLVwidth=ch_sLVw.value,
        mLVheight=ch_mLVh.value, stdevLVheight=ch_sLVh.value,
        mLVasym=ch_mLVa.value, stdevLVasym=ch_sLVa.value,
        mLVthin=ch_mLVt.value, stdevLVthin=ch_sLVt.value,
        mCSnum=ch_mCSn.value, stdevCSnum=ch_sCSn.value,
        mCSnumlobe=ch_mCSnl.value, stdevCSnumlobe=ch_sCSnl.value,
        mCSsource=ch_mCSs.value, stdevCSsource=ch_sCSs.value,
        mCSLOLL=ch_mCSLL.value, stdevCSLOLL=ch_sCSLL.value,
        mCSLOWW=ch_mCSWW.value, stdevCSLOWW=ch_sCSWW.value,
        mCSLOl=ch_mCSl.value, stdevCSLOl=ch_sCSl.value,
        mCSLOw=ch_mCSw.value, stdevCSLOw=ch_sCSw.value,
        mCSLO_hwratio=ch_mCShw.value, stdevCSLO_hwratio=ch_sCShw.value,
        mCSLO_dwratio=ch_mCSdw.value, stdevCSLO_dwratio=ch_sCSdw.value,
        mFFCHprop=ch_mFFCH.value, stdevFFCHprop=ch_sFFCH.value,
        Cf=ch_Cf.value, scour_factor=ch_scour.value, gradient=ch_grad.value, Q=ch_Q.value,
        CHndraw=ch_CHndraw.value, ndiscr=ch_ndiscr.value, nCHcor=ch_nCHcor.value,
        azimuth=ch_azim.value,
        output_facies=ch_outmode.value,
        seed=int(w_seed.value),
    )

# Re-entrancy guard: prevent overlapping generations if user clicks twice
_generating = {'busy': False}

def _build_layer_and_fig():
    """Run the simulation and build the figure. Returns (fig, status_html)."""
    model_type = model_selector.value
    grid_kw = dict(
        nx=w_nx.value, ny=w_ny.value, nz=w_nz.value,
        x_len=w_xlen.value, y_len=w_ylen.value, z_len=w_zlen.value,
        top_depth=w_depth.value, dip=w_dip.value)
    if model_type == 'Lobe':
        layer = gr.LobeLayer(**grid_kw)
        layer.create_geology(
            poro_ave=l_poro.value, perm_ave=l_perm.value,
            poro_std=l_poro_std.value, perm_std=l_perm_std.value, ntg=l_ntg.value,
            dhmin=l_dhmin.value, dhmax=l_dhmax.value,
            rmin=l_rmin.value, rmax=l_rmax.value,
            asp=l_asp.value, theta0=l_theta.value, m=l_m.value,
            upthinning=l_upthin.value, bouma_factor=l_bouma.value)
    elif model_type == 'Gaussian':
        layer = gr.GaussianLayer(**grid_kw)
        layer.create_geology(
            poro_ave=g_poro.value, perm_ave=g_perm.value,
            poro_std=g_poro_std.value, perm_std=g_perm_std.value, ntg=g_ntg.value,
            facies_filter=(g_fx.value, g_fy.value, g_fz.value),
            sand_filter=(g_sx.value, g_sy.value, g_sz.value),
            nugget=g_nugget.value, poro_perm_corr=g_corr.value)
    elif model_type == 'Meandering Channel':
        layer = gr.MeanderingChannelLayer(**grid_kw)
        layer.create_geology(**_channel_kwargs())
    elif model_type == 'Braided Channel':
        layer = gr.BraidedChannelLayer(**grid_kw)
        layer.create_geology(**_channel_kwargs())
    elif model_type == 'Delta':
        layer = gr.DeltaLayer(**grid_kw)
        kmin = min(d_kmin.value, d_kmax.value)
        kmax = max(d_kmin.value, d_kmax.value)
        layer.create_geology(
            feeder_width=d_feeder.value, n_generations=d_ngen.value,
            fan_angle_deg=d_fan.value, bifurcation_depth=d_bif.value,
            children_per_split=(kmin, kmax), fan_asymmetry=d_asym.value,
            azimuth=d_azim.value, apex_x_fraction=d_apex.value,
            progradation_fraction=d_prog.value, trunk_length_factor=d_trunk.value,
            length_taper=d_ltap.value, width_taper=d_wtap.value, meander_amp=d_mwig.value,
            mouth_bar_length_factor=d_mbL.value, mouth_bar_width_factor=d_mbW.value,
            mouth_bar_hw_ratio=d_mbhw.value, mouth_bar_dw_ratio=d_mbdw.value,
            dwratio=d_dwr.value, poro_ave=d_poro.value)

    # Choose property
    prop = w_prop.value
    if prop == 'Porosity':
        data = layer.poro_mat; label = f'{model_type} — Porosity'
        cmap = 'resmill'; norm = None; mask_z = True
    elif prop == 'Permeability':
        data = layer.perm_mat; label = f'{model_type} — Permeability (mD)'
        cmap = 'resmill'; norm = None; mask_z = True
    elif prop == 'Facies (binary)':
        if hasattr(layer, 'facies'):
            binary = (layer.facies >= 1).astype(float) if layer.facies.min() < 0 else layer.facies.astype(float)
        else:
            binary = layer.active.astype(float)
        data = binary; label = f'{model_type} — Facies (binary)'
        cmap = 'resmill'; norm = None; mask_z = False
    else:  # Facies (Alluvsim 6-class)
        if hasattr(layer, 'facies_alluvsim'):
            data = layer.facies_alluvsim.astype(float)
        elif hasattr(layer, 'facies'):
            data = layer.facies.astype(float)
        else:
            data = layer.active.astype(float)
        cmap, norm = alluvsim_cmap()
        label = f'{model_type} — Alluvsim 6-class facies'; mask_z = False

    view = w_view.value
    plt.close('all')  # avoid figure-leak between regenerations
    if cmap == 'resmill':
        if view == 'Cube Slices':
            gr.plot_cube_slices(data, title=label, mask_zeros=mask_z)
        else:
            axis = {'Z Slices': 2, 'X Slices': 0, 'Y Slices': 1}[view]
            gr.plot_slices(data, axis=axis, title=label, mask_zeros=mask_z)
        fig = plt.gcf()
    else:
        # Alluvsim 6-class — manual cmap with legend
        axis = {'Cube Slices': 2, 'Z Slices': 2, 'X Slices': 0, 'Y Slices': 1}[view]
        ncols = 4
        indices = np.linspace(0, data.shape[axis] - 1, 8, dtype=int)
        nrows = (len(indices) + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3 * nrows))
        axes = np.array(axes).reshape(-1)
        for i, idx in enumerate(indices):
            if axis == 0: sl = data[idx, :, :].T
            elif axis == 1: sl = data[:, idx, :].T
            else: sl = data[:, :, idx].T
            axes[i].imshow(sl, cmap=cmap, norm=norm, origin='lower', interpolation='nearest')
            axes[i].set_title(f'{["X","Y","Z"][axis]}={idx}', fontsize=10)
        for j in range(len(indices), len(axes)):
            axes[j].axis('off')
        handles = [plt.Rectangle((0, 0), 1, 1, fc=ALLUVSIM_FACIES_COLORS[c]) for c in sorted(ALLUVSIM_FACIES_NAMES.keys())]
        labels = [ALLUVSIM_FACIES_NAMES[c] for c in sorted(ALLUVSIM_FACIES_NAMES.keys())]
        fig.legend(handles, labels, loc='lower center', ncol=6, bbox_to_anchor=(0.5, -0.02), fontsize=9)
        fig.suptitle(label, fontsize=12)
        fig.tight_layout()

    # Build status string
    parts = [f'<b>Grid</b> ({w_nx.value}×{w_ny.value}×{w_nz.value})',
             f'<b>NTG</b> {layer.active.mean()*100:.1f}%']
    if hasattr(layer, 'facies_alluvsim'):
        fa = layer.facies_alluvsim
        for c in sorted(ALLUVSIM_FACIES_NAMES.keys()):
            frac = (fa == c).mean() * 100
            if frac > 0.05:
                colour = ALLUVSIM_FACIES_COLORS[c]
                name = ALLUVSIM_FACIES_NAMES[c].split()[0]
                parts.append(f'<span style="color:{colour}">■</span>&nbsp;{name}={frac:.1f}%')
    return fig, '&nbsp;&nbsp;'.join(parts)

def generate_model(*args, force=False):
    """Generate and render the current model.

    Skips silently when called via a slider observer with auto-mode off
    (lets you drag sliders without triggering re-generation). The
    'Generate' button and preset selector pass force=True to bypass.
    """
    if _applying_preset['lock']:
        return
    if not force and not w_auto.value:
        return  # manual mode: ignore observer-triggered calls
    if _generating['busy']:
        return  # debounce overlapping clicks
    _generating['busy'] = True
    try:
        # Immediate visible feedback BEFORE the simulation runs
        status_label.value = (
            f'<i style="color:#0a6">⏳ Generating {model_selector.value}…</i>')
        t0 = time.perf_counter()
        try:
            fig, stats_html = _build_layer_and_fig()
        except Exception as e:
            import traceback
            tb = traceback.format_exc()
            status_label.value = f'<span style="color:red">Error: {e}</span>'
            output_area.clear_output(wait=False)
            with output_area:
                print(tb)
            return
        dt = time.perf_counter() - t0
        # Replace the output in one shot — clear, then display the figure
        output_area.clear_output(wait=True)
        with output_area:
            display(fig)
            plt.close(fig)  # release the figure now that it's rendered
        status_label.value = (
            f'<span style="color:#0a6">✓ Done in {dt:.2f}s.</span>&nbsp;&nbsp;{stats_html}')
    finally:
        _generating['busy'] = False

# Generate button always renders, regardless of auto-mode toggle
w_gen_btn.on_click(lambda b: generate_model(force=True))

# Wire all widgets to auto-generate (no-op when auto-mode is off — see
# generate_model's `if not force and not w_auto.value: return` guard).
channel_widgets = [
    ch_nlevel, ch_lvl_z, ch_NTG, ch_ntime, ch_pAvOut, ch_pAvIn,
    ch_mDepth, ch_sDepth, ch_sDepth2, ch_mWdr, ch_sWdr, ch_mSinu, ch_sSinu,
    ch_mAzi, ch_sAzi, ch_mSrc, ch_sSrc, ch_mMig, ch_sMig,
    ch_mLVd, ch_sLVd, ch_mLVw, ch_sLVw, ch_mLVh, ch_sLVh, ch_mLVa, ch_sLVa, ch_mLVt, ch_sLVt,
    ch_mCSn, ch_sCSn, ch_mCSnl, ch_sCSnl, ch_mCSs, ch_sCSs,
    ch_mCSLL, ch_sCSLL, ch_mCSWW, ch_sCSWW, ch_mCSl, ch_sCSl, ch_mCSw, ch_sCSw,
    ch_mCShw, ch_sCShw, ch_mCSdw, ch_sCSdw, ch_mFFCH, ch_sFFCH,
    ch_Cf, ch_scour, ch_grad, ch_Q, ch_CHndraw, ch_ndiscr, ch_nCHcor,
    ch_azim, ch_outmode,
]
all_widgets = (
    [model_selector, w_nx, w_ny, w_nz, w_xlen, w_ylen, w_zlen, w_depth, w_dip, w_seed, w_view, w_prop]
    + [l_poro, l_perm, l_poro_std, l_perm_std, l_ntg, l_dhmin, l_dhmax, l_rmin, l_rmax, l_asp, l_theta, l_m, l_upthin, l_bouma]
    + [g_poro, g_perm, g_poro_std, g_perm_std, g_ntg, g_fx, g_fy, g_fz, g_sx, g_sy, g_sz, g_nugget, g_corr]
    + channel_widgets
    + [d_feeder, d_ngen, d_fan, d_bif, d_kmin, d_kmax, d_asym, d_azim, d_apex, d_prog,
       d_trunk, d_ltap, d_wtap, d_mwig, d_mbL, d_mbW, d_mbhw, d_mbdw, d_dwr, d_poro]
)
for w in all_widgets:
    w.observe(generate_model, names='value')

update_params()
display(widgets.VBox([
    widgets.HTML('<h2>ResMill Interactive Explorer — Alluvsim Edition</h2>'),
    widgets.HTML('<i>Drag sliders freely; click <b>Generate</b> to render. '
                 'Tick <b>Auto-regenerate</b> for live updates on every change.</i>'),
    model_selector, grid_box, param_area, view_box, gen_box, output_area,
]))